# YOLO용 데이터셋 만들기
여기서는 앞서 전처리한 COCO 형식의 JSON 파일들을 활용하여 YOLO 학습용 데이터를 만들고, 실제 학습을 하여 결과를 확인한다.</br>
그 첫 단계로, 우선 기본 dataset과 추가 dataset의 annotation 파일들을 각각 합쳐준다.

In [1]:
import os
import json
import shutil
import copy
from sklearn.model_selection import train_test_split
from pycocotools.coco import COCO
import pandas as pd
from PIL import Image
from glob import glob
from tqdm import tqdm
from ultralytics import YOLO
import numpy as np
import cv2

In [2]:
cat_id_to_name={
    1899: "보령부스파정 5mg",
    2482: "뮤테란캡슐 100mg",
    3350: "일양하이트린정 2mg",
    3482: "기넥신에프정(은행엽엑스)(수출용)",
    3543: "무코스타정(레바미피드)(비매품)",
    3742: "알드린정",
    3831: "뉴로메드정(옥시라세탐)",
    4377: "타이레놀정500mg",
    4542: "에어탈정(아세클로페낙)",
    5093: "삼남건조수산화알루미늄겔정",
    5885: "타이레놀이알서방정(아세트아미노펜)(수출용)",
    6191: "삐콤씨에프정 618.6mg/병",
    6562: "조인스정 200mg",
    10220: "쎄로켈정 100mg",
    10223: "넥시움정 40mg",
    12080: "리렉스펜정 300mg/PTP",
    12246: "아빌리파이정 10mg",
    12419: "자이프렉사정 2.5mg",
    12777: "다보타민큐정 10mg/병",
    13394: "써스펜8시간이알서방정 650mg",
    13899: "에빅사정(메만틴염산염)(비매품)",
    16231: "리피토정 20mg",
    16261: "크레스토정 20mg",
    16547: "가바토파정 100mg",
    16550: "동아가바펜틴정 800mg",
    16687: "오마코연질캡슐(오메가-3-산에틸에스테르90)",
    18109: "란스톤엘에프디티정 30mg",
    18146: "리리카캡슐 150mg",
    18356: "종근당글리아티린연질캡슐(콜린알포세레이트)",
    19231: "콜리네이트연질캡슐 400mg",
    19551: "트루비타정 60mg/병",
    19606: "스토가정 10mg",
    19860: "노바스크정 5mg",
    20013: "마도파정",
    20237: "플라빅스정 75mg",
    20876: "엑스포지정 5/160mg",
    21025: "펠루비정(펠루비프로펜)",
    21324: "아토르바정 10mg",
    21770: "라비에트정 20mg",
    22073: "리피로우정 20mg",
    22346: "자누비아정 50mg",
    22361: "맥시부펜이알정 300mg",
    22626: "메가파워정 90mg/병",
    23202: "쿠에타핀정 25mg",
    23222: "비타비백정 100mg/병",
    24849: "놀텍정 10mg",
    25366: "자누메트정 50/850mg",
    25437: "큐시드정 31.5mg/PTP",
    25468: "아모잘탄정 5/100mg",
    27652: "세비카정 10/40mg",
    27732: "트윈스타정 40/5mg",
    27776: "카나브정 60mg",
    27925: "울트라셋이알서방정",
    27992: "졸로푸트정 100mg",
    28762: "트라젠타정(리나글립틴)",
    29344: "비모보정 500/20mg",
    29450: "레일라정",
    29666: "리바로정 4mg",
    29870: "렉사프로정 15mg",
    30307: "트라젠타듀오정 2.5/850mg",
    31704: "낙소졸정 500/20mg",
    31862: "아질렉트정(라사길린메실산염)",
    31884: "자누메트엑스알서방정 100/1000mg",
    32309: "글리아타민연질캡슐",
    33008: "신바로정",
    33207: "에스원엠프정 20mg",
    33877: "브린텔릭스정 20mg",
    33879: "글리틴정(콜린알포세레이트)",
    34596: "제미메트서방정 50/1000mg",
    35205: "아토젯정 10/40mg",
    36636: "로수젯정10/5밀리그램",
    38161: "로수바미브정 10/20mg",
    41767: "카발린캡슐 25mg",
    44198: "케이캡정 50mg"
}

In [3]:
cat_id_to_yolo_cat={cat:idx for idx,cat in enumerate(cat_id_to_name.keys())}
cat_id_to_yolo_cat

{1899: 0,
 2482: 1,
 3350: 2,
 3482: 3,
 3543: 4,
 3742: 5,
 3831: 6,
 4377: 7,
 4542: 8,
 5093: 9,
 5885: 10,
 6191: 11,
 6562: 12,
 10220: 13,
 10223: 14,
 12080: 15,
 12246: 16,
 12419: 17,
 12777: 18,
 13394: 19,
 13899: 20,
 16231: 21,
 16261: 22,
 16547: 23,
 16550: 24,
 16687: 25,
 18109: 26,
 18146: 27,
 18356: 28,
 19231: 29,
 19551: 30,
 19606: 31,
 19860: 32,
 20013: 33,
 20237: 34,
 20876: 35,
 21025: 36,
 21324: 37,
 21770: 38,
 22073: 39,
 22346: 40,
 22361: 41,
 22626: 42,
 23202: 43,
 23222: 44,
 24849: 45,
 25366: 46,
 25437: 47,
 25468: 48,
 27652: 49,
 27732: 50,
 27776: 51,
 27925: 52,
 27992: 53,
 28762: 54,
 29344: 55,
 29450: 56,
 29666: 57,
 29870: 58,
 30307: 59,
 31704: 60,
 31862: 61,
 31884: 62,
 32309: 63,
 33008: 64,
 33207: 65,
 33877: 66,
 33879: 67,
 34596: 68,
 35205: 69,
 36636: 70,
 38161: 71,
 41767: 72,
 44198: 73}

In [4]:
base_path='additional_training_data'
annotation_root='train_annotations'
folders=sorted(os.listdir(os.path.join(base_path,annotation_root)))
for folder in folders:
    subs=sorted(os.listdir(os.path.join(base_path,annotation_root,folder)))
    for sub in subs:
        if os.path.isdir(os.path.join(base_path,annotation_root,folder,sub)):
            os.system(f'rm -rf {os.path.join(base_path,annotation_root,folder,sub)}')

In [5]:
output_all_coco = {
    "images": [],
    "annotations": [],
    "categories": []
}
image_id_max=0
annot_id_max=0
base_path='additional_training_data'
annotation_root='train_annotations'
folders=sorted(os.listdir(os.path.join(base_path,annotation_root)))
for folder in folders:
    files=copy.deepcopy([entry.name for entry in os.scandir(os.path.join(base_path,annotation_root,folder)) if entry.is_file() and entry.name.endswith(".json")])
    for file in files:
        with open(os.path.join(base_path,annotation_root,folder,file), "r", encoding="utf-8") as f:
            data=json.load(f)
            for image in data['images']:
                image_id_max=max(image['id'],image_id_max)
                output_all_coco['images'].append(image)
            for annot in data['annotations']:
                annot_id_max=max(annot['id'],annot_id_max)
                annot['category_id']=cat_id_to_yolo_cat[annot['category_id']]
                output_all_coco['annotations'].append(annot)
            for cat in data['categories']:
                cat['id']=cat_id_to_yolo_cat[cat['id']]
                output_all_coco['categories'].append(cat)
t=pd.DataFrame(output_all_coco['categories'])
t=t.drop_duplicates()
output_all_coco['categories']=t.to_dict(orient='records')

save_path = os.path.join(base_path, "train.json")
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(output_all_coco, f, ensure_ascii=False, indent=4)

print("train.json 생성됨:", save_path)

train.json 생성됨: additional_training_data/train.json


In [6]:
output_all_coco = {
    "images": [],
    "annotations": [],
    "categories": []
}
base_path='ai06-level1-project'
annotation_root='train_annotations'
folders=sorted(os.listdir(os.path.join(base_path,annotation_root)))
for folder in folders:
    files=copy.deepcopy([entry.name for entry in os.scandir(os.path.join(base_path,annotation_root,folder)) if entry.is_file() and entry.name.endswith(".json")])
    for file in files:
        with open(os.path.join(base_path,annotation_root,folder,file), "r", encoding="utf-8") as f:
            data=json.load(f)
            image_ids=[]
            for image in data['images']:
                image['id']=image['id']+image_id_max
                image_ids.append(image['id'])
                output_all_coco['images'].append(image)
            for annot in data['annotations']:
                annot['id']=annot['id']+annot_id_max
                annot['image_id']=annot['image_id']+image_id_max
                assert annot['image_id'] in image_ids
                annot['category_id']=cat_id_to_yolo_cat[annot['category_id']]
                output_all_coco['annotations'].append(annot)
            for cat in data['categories']:
                cat['id']=cat_id_to_yolo_cat[cat['id']]
                output_all_coco['categories'].append(cat)

t=pd.DataFrame(output_all_coco['categories'])
t=t.drop_duplicates()
output_all_coco['categories']=t.to_dict(orient='records')

save_path = os.path.join(base_path, "train.json")
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(output_all_coco, f, ensure_ascii=False, indent=4)

print("train.json 생성됨:", save_path)

train.json 생성됨: ai06-level1-project/train.json


이제 두 파일을 합쳐준다.

In [7]:
#YOLO용으로 데이터 전처리

BASE1 = "./ai06-level1-project/"
BASE2 = "./additional_training_data/"
IMG_DIR1 = os.path.join(BASE1, "train_output")
IMG_DIR2 = os.path.join(BASE2, "train_cleaned")
ANN_FILE1 = os.path.join("./ai06-level1-project/", "train.json")
ANN_FILE2 = os.path.join("./additional_training_data/", "train.json")
TEST_IMG_DIR = os.path.join(BASE1, "test_images")

OUT_DIR = "yolo_dataset"

os.makedirs(os.path.join(OUT_DIR, "images/train"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "images/val"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "labels/train"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "labels/val"), exist_ok=True)

with open(ANN_FILE1, "r", encoding="utf-8") as f:
    dataset1 = json.load(f)

with open(ANN_FILE2, "r", encoding="utf-8") as f:
    dataset2 = json.load(f)

i=pd.DataFrame(dataset1["images"] + dataset2["images"])
i=i.drop_duplicates()
i=i.to_dict(orient='records')

cat=pd.DataFrame(dataset1["categories"] + dataset2["categories"])
cat=cat.drop_duplicates()
cat=cat.to_dict(orient='records')

tt=dataset1["annotations"] + dataset2["annotations"]

dataset = {
    "images": i,
    "annotations": dataset1["annotations"] + dataset2["annotations"],
    "categories": cat
}

dataset['categories']

[{'id': 60, 'supercategory': 'pill', 'name': '낙소졸정 500/20mg'},
 {'id': 24, 'supercategory': 'pill', 'name': '동아가바펜틴정 800mg'},
 {'id': 14, 'supercategory': 'pill', 'name': '넥시움정 40mg'},
 {'id': 0, 'supercategory': 'pill', 'name': '보령부스파정 5mg'},
 {'id': 64, 'supercategory': 'pill', 'name': '신바로정'},
 {'id': 36, 'supercategory': 'pill', 'name': '펠루비정(펠루비프로펜)'},
 {'id': 23, 'supercategory': 'pill', 'name': '가바토파정 100mg'},
 {'id': 26, 'supercategory': 'pill', 'name': '란스톤엘에프디티정 30mg'},
 {'id': 52, 'supercategory': 'pill', 'name': '울트라셋이알서방정'},
 {'id': 55, 'supercategory': 'pill', 'name': '비모보정 500/20mg'},
 {'id': 56, 'supercategory': 'pill', 'name': '레일라정'},
 {'id': 31, 'supercategory': 'pill', 'name': '스토가정 10mg'},
 {'id': 38, 'supercategory': 'pill', 'name': '라비에트정 20mg'},
 {'id': 45, 'supercategory': 'pill', 'name': '놀텍정 10mg'},
 {'id': 65, 'supercategory': 'pill', 'name': '에스원엠프정 20mg'},
 {'id': 73, 'supercategory': 'pill', 'name': '케이캡정 50mg'},
 {'id': 7, 'supercategory': 'pill', 'name'

In [8]:
len(dataset['categories'])

74

잘 합쳐진듯 하니 이제 데이터셋 생성을 수행한다.

In [9]:
#YOLO용으로 데이터 전처리

OUT_DIR = "./yolo_dataset"

os.makedirs(os.path.join(OUT_DIR, "images/train"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "images/val"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "labels/train"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "labels/val"), exist_ok=True)

coco = COCO()
coco.dataset = dataset
coco.createIndex()

img_ids = list(coco.imgs.keys())

train_ids, val_ids = train_test_split(img_ids, test_size=0.2, random_state=42)

creating index...
index created!


In [10]:
coco.cats

{60: {'id': 60, 'supercategory': 'pill', 'name': '낙소졸정 500/20mg'},
 24: {'id': 24, 'supercategory': 'pill', 'name': '동아가바펜틴정 800mg'},
 14: {'id': 14, 'supercategory': 'pill', 'name': '넥시움정 40mg'},
 0: {'id': 0, 'supercategory': 'pill', 'name': '보령부스파정 5mg'},
 64: {'id': 64, 'supercategory': 'pill', 'name': '신바로정'},
 36: {'id': 36, 'supercategory': 'pill', 'name': '펠루비정(펠루비프로펜)'},
 23: {'id': 23, 'supercategory': 'pill', 'name': '가바토파정 100mg'},
 26: {'id': 26, 'supercategory': 'pill', 'name': '란스톤엘에프디티정 30mg'},
 52: {'id': 52, 'supercategory': 'pill', 'name': '울트라셋이알서방정'},
 55: {'id': 55, 'supercategory': 'pill', 'name': '비모보정 500/20mg'},
 56: {'id': 56, 'supercategory': 'pill', 'name': '레일라정'},
 31: {'id': 31, 'supercategory': 'pill', 'name': '스토가정 10mg'},
 38: {'id': 38, 'supercategory': 'pill', 'name': '라비에트정 20mg'},
 45: {'id': 45, 'supercategory': 'pill', 'name': '놀텍정 10mg'},
 65: {'id': 65, 'supercategory': 'pill', 'name': '에스원엠프정 20mg'},
 73: {'id': 73, 'supercategory': 'pill', '

In [11]:
len(img_ids)

651

In [12]:
def convert_to_yolo_bbox(box, img_w, img_h):
    x, y, w, h = box
    cx = (x + w/2) / img_w
    cy = (y + h/2) / img_h
    w /= img_w
    h /= img_h
    return cx, cy, w, h


def process_image(img_id, split="train"):

    img_info = coco.loadImgs(img_id)[0]
    file_name = img_info["file_name"]
    width, height = img_info["width"], img_info["height"]

    src_img_path1 = os.path.join(IMG_DIR1, file_name)
    src_img_path2 = os.path.join(IMG_DIR2, file_name)
    dst_img_path = os.path.join(OUT_DIR, f"images/{split}/{file_name}")

    if os.path.exists(src_img_path1):
        shutil.copy(src_img_path1, dst_img_path)
    elif os.path.exists(src_img_path2):
        shutil.copy(src_img_path2, dst_img_path)
    else:
        print("이미지 없음")
        return

    label_path = os.path.join(OUT_DIR, f"labels/{split}/{file_name.replace('.png', '.txt')}")

    ann_ids = coco.getAnnIds(imgIds=img_id)
    anns = coco.loadAnns(ann_ids)

    with open(label_path, "w", encoding="utf-8") as f:
        for ann in anns:
            yolo_class = ann["category_id"]
            bbox = ann["bbox"]
            yolo_box = convert_to_yolo_bbox(bbox, width, height)

            f.write(f"{yolo_class} {' '.join([str(round(v, 6)) for v in yolo_box])}\n")


for img_id in train_ids:
    process_image(img_id, split="train")

for img_id in val_ids:
    process_image(img_id, split="val")

print("YOLO dataset 생성 완료")


yaml_path = os.path.join(OUT_DIR, "data.yaml")
num_classes = len(coco.cats)
names = [coco.cats[k]["name"] for k in sorted(coco.cats.keys())]

with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(f"path: {OUT_DIR}\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n\n")
    f.write(f"nc: {num_classes}\n")
    f.write(f"names: {names}\n")

print("data.yaml 파일 생성 완료")

YOLO dataset 생성 완료
data.yaml 파일 생성 완료


# YOLO 학습하기
앞서 완성한 데이터셋을 활용하여 YOLO 모델을 학습한다.

In [13]:
model = YOLO("yolo11x.pt")

In [14]:
model.train(
    data=r"./yolo_dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=4,
    lr0=1e-3,
    lrf=0.2,
    device=0,
    workers=2,
    amp=True,
    name="pill_augment_yolo11x",
    pretrained=True,
    seed=42,
    optimizer='Adam',
    project='./ai06-level1-project/train_checkpoints',
    save=True,
    exist_ok=True,
    plots=True,
    degrees=45,
    mosaic=0,
    translate=0.2,
    shear=10,
    perspective=0.0001
)

New https://pypi.org/project/ultralytics/8.3.240 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.235 🚀 Python-3.11.14 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./yolo_dataset/data.yaml, degrees=45, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.2, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11x.pt, momentum=0.937, mosaic=0, multi_scale=False, name=pill_augment_yolo11x, n

/home/codeitDev/miniconda3/envs/codeit/lib/python3.11/site-packages/ultralytics/utils/metrics.py:567: UserWarning: Glyph 48372 (\N{HANGUL SYLLABLE BO}) missing from font(s) DejaVu Sans.
  fig.savefig(plot_fname, dpi=250)
/home/codeitDev/miniconda3/envs/codeit/lib/python3.11/site-packages/ultralytics/utils/metrics.py:567: UserWarning: Glyph 47161 (\N{HANGUL SYLLABLE RYEONG}) missing from font(s) DejaVu Sans.
  fig.savefig(plot_fname, dpi=250)
/home/codeitDev/miniconda3/envs/codeit/lib/python3.11/site-packages/ultralytics/utils/metrics.py:567: UserWarning: Glyph 48512 (\N{HANGUL SYLLABLE BU}) missing from font(s) DejaVu Sans.
  fig.savefig(plot_fname, dpi=250)
/home/codeitDev/miniconda3/envs/codeit/lib/python3.11/site-packages/ultralytics/utils/metrics.py:567: UserWarning: Glyph 49828 (\N{HANGUL SYLLABLE SEU}) missing from font(s) DejaVu Sans.
  fig.savefig(plot_fname, dpi=250)
/home/codeitDev/miniconda3/envs/codeit/lib/python3.11/site-packages/ultralytics/utils/metrics.py:567: UserWarni

                   all       1880       5588      0.996      0.999      0.995      0.942
            보령부스파정 5mg         82         82      0.998          1      0.995      0.933
           뮤테란캡슐 100mg         84         84      0.997          1      0.995      0.954
           일양하이트린정 2mg         64         64      0.998          1      0.995       0.98
    기넥신에프정(은행엽엑스)(수출용)         84         84      0.998          1      0.995      0.926
     무코스타정(레바미피드)(비매품)         59         59      0.996          1      0.995      0.946
                  알드린정        102        102      0.999          1      0.995      0.913
          뉴로메드정(옥시라세탐)         57         57      0.998          1      0.995      0.913
            타이레놀정500mg         92         92      0.999          1      0.995      0.934
          에어탈정(아세클로페낙)         71         71      0.997          1      0.995      0.966
         삼남건조수산화알루미늄겔정         58         58      0.997          1      0.995      0.962
타이레놀이알서방정(아세트아미노펜)(수출

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a9c19d7cb50>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026, 

In [15]:
model=YOLO('./ai06-level1-project/train_checkpoints/pill_augment_yolo11x/weights/last.pt')
model.train(
    data=r"./yolo_dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=4,
    lr0=1e-4,
    lrf=0.05,
    device=0,
    workers=2,
    amp=True,
    name="pill_augment_yolo11x_2",
    pretrained=True,
    seed=42,
    optimizer='Adam',
    project='./ai06-level1-project/train_checkpoints',
    save=True,
    exist_ok=True,
    degrees=45,
    mosaic=0,
    translate=0.2,
    shear=10,
    perspective=0.0001
)

New https://pypi.org/project/ultralytics/8.3.240 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.235 🚀 Python-3.11.14 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./yolo_dataset/data.yaml, degrees=45, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.05, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=./ai06-level1-project/train_checkpoints/pill_augment_yolo11x/weights/last.pt, momen

/home/codeitDev/miniconda3/envs/codeit/lib/python3.11/site-packages/ultralytics/utils/metrics.py:567: UserWarning: Glyph 48372 (\N{HANGUL SYLLABLE BO}) missing from font(s) DejaVu Sans.
  fig.savefig(plot_fname, dpi=250)
/home/codeitDev/miniconda3/envs/codeit/lib/python3.11/site-packages/ultralytics/utils/metrics.py:567: UserWarning: Glyph 47161 (\N{HANGUL SYLLABLE RYEONG}) missing from font(s) DejaVu Sans.
  fig.savefig(plot_fname, dpi=250)
/home/codeitDev/miniconda3/envs/codeit/lib/python3.11/site-packages/ultralytics/utils/metrics.py:567: UserWarning: Glyph 48512 (\N{HANGUL SYLLABLE BU}) missing from font(s) DejaVu Sans.
  fig.savefig(plot_fname, dpi=250)
/home/codeitDev/miniconda3/envs/codeit/lib/python3.11/site-packages/ultralytics/utils/metrics.py:567: UserWarning: Glyph 49828 (\N{HANGUL SYLLABLE SEU}) missing from font(s) DejaVu Sans.
  fig.savefig(plot_fname, dpi=250)
/home/codeitDev/miniconda3/envs/codeit/lib/python3.11/site-packages/ultralytics/utils/metrics.py:567: UserWarni

                   all       1880       5588      0.998      0.999      0.995      0.988
            보령부스파정 5mg         82         82          1          1      0.995      0.969
           뮤테란캡슐 100mg         84         84      0.999          1      0.995       0.98
           일양하이트린정 2mg         64         64      0.999          1      0.995      0.995
    기넥신에프정(은행엽엑스)(수출용)         84         84      0.999          1      0.995      0.993
     무코스타정(레바미피드)(비매품)         59         59          1          1      0.995      0.995
                  알드린정        102        102      0.999          1      0.995      0.995
          뉴로메드정(옥시라세탐)         57         57      0.999          1      0.995      0.993
            타이레놀정500mg         92         92      0.999          1      0.995       0.98
          에어탈정(아세클로페낙)         71         71      0.999          1      0.995      0.995
         삼남건조수산화알루미늄겔정         58         58      0.999          1      0.995      0.995
타이레놀이알서방정(아세트아미노펜)(수출

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a9b60d8de90>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026, 

In [16]:
model=YOLO('./ai06-level1-project/train_checkpoints/pill_augment_yolo11x_2/weights/best.pt')

In [17]:
model.predict(
    source="./ai06-level1-project/test_cleaned",
    save=True,
    save_txt=True,
    save_conf=True,
    half=True,
    amp=True
)


image 1/843 /home/codeitDev/project/Basic_Project_team_2/ai06-level1-project/test_cleaned/1.png: 640x512 1 보령부스파정 5mg, 1 동아가바펜틴정 800mg, 1 놀텍정 10mg, 1 울트라셋이알서방정, 675.4ms
image 2/843 /home/codeitDev/project/Basic_Project_team_2/ai06-level1-project/test_cleaned/10.png: 640x512 1 보령부스파정 5mg, 1 가바토파정 100mg, 1 라비에트정 20mg, 1 레일라정, 121.7ms
image 3/843 /home/codeitDev/project/Basic_Project_team_2/ai06-level1-project/test_cleaned/100.png: 640x512 1 보령부스파정 5mg, 1 가바토파정 100mg, 1 란스톤엘에프디티정 30mg, 1 신바로정, 63.4ms
image 4/843 /home/codeitDev/project/Basic_Project_team_2/ai06-level1-project/test_cleaned/1003.png: 640x512 1 기넥신에프정(은행엽엑스)(수출용), 1 리피토정 20mg, 1 트윈스타정 40/5mg, 1 제미메트서방정 50/1000mg, 63.1ms
image 5/843 /home/codeitDev/project/Basic_Project_team_2/ai06-level1-project/test_cleaned/1004.png: 640x512 1 기넥신에프정(은행엽엑스)(수출용), 1 리피토정 20mg, 1 트윈스타정 40/5mg, 1 제미메트서방정 50/1000mg, 63.3ms
image 6/843 /home/codeitDev/project/Basic_Project_team_2/ai06-level1-project/test_cleaned/1005.png: 640x512 1 기넥신에프정(은행엽엑스

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: '보령부스파정 5mg', 1: '뮤테란캡슐 100mg', 2: '일양하이트린정 2mg', 3: '기넥신에프정(은행엽엑스)(수출용)', 4: '무코스타정(레바미피드)(비매품)', 5: '알드린정', 6: '뉴로메드정(옥시라세탐)', 7: '타이레놀정500mg', 8: '에어탈정(아세클로페낙)', 9: '삼남건조수산화알루미늄겔정', 10: '타이레놀이알서방정(아세트아미노펜)(수출용)', 11: '삐콤씨에프정 618.6mg/병', 12: '조인스정 200mg', 13: '쎄로켈정 100mg', 14: '넥시움정 40mg', 15: '리렉스펜정 300mg/PTP', 16: '아빌리파이정 10mg', 17: '자이프렉사정 2.5mg', 18: '다보타민큐정 10mg/병', 19: '써스펜8시간이알서방정 650mg', 20: '에빅사정(메만틴염산염)(비매품)', 21: '리피토정 20mg', 22: '크레스토정 20mg', 23: '가바토파정 100mg', 24: '동아가바펜틴정 800mg', 25: '오마코연질캡슐(오메가-3-산에틸에스테르90)', 26: '란스톤엘에프디티정 30mg', 27: '리리카캡슐 150mg', 28: '종근당글리아티린연질캡슐(콜린알포세레이트)', 29: '콜리네이트연질캡슐 400mg', 30: '트루비타정 60mg/병', 31: '스토가정 10mg', 32: '노바스크정 5mg', 33: '마도파정', 34: '플라빅스정 75mg', 35: '엑스포지정 5/160mg', 36: '펠루비정(펠루비프로펜)', 37: '아토르바정 10mg', 38: '라비에트정 20mg', 39: '리피로우정 20mg', 40: '자누비아정 50mg', 41: '맥시부펜이알정 300mg', 42:

In [35]:
results = model.val(
    data="yolo_dataset/data.yaml",
    device=0,
    workers=2
)
print("F1 score curve:", results.box.f1)

Ultralytics 8.3.235 🚀 Python-3.11.14 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLOv10x summary (fused): 192 layers, 29,467,790 parameters, 0 gradients
val: Fast image access ✅ (ping: 2.5±0.4 ms, read: 57.7±14.6 MB/s, size: 367.1 KB)
val: Scanning /home/codeitDev/project/Basic_Project_team_2/yolo_dataset/labels/val.cache... 1652 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1652/1652 1.6Mit/s 0.0s0s
val: /home/codeitDev/project/Basic_Project_team_2/yolo_dataset/images/val/K-000250-002483-012081-019552_0_2_0_2_70_000_200.png: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 104/104 2.0it/s 51.7s0.5s


/home/codeitDev/miniconda3/envs/codeit/lib/python3.11/site-packages/ultralytics/utils/metrics.py:567: UserWarning: Glyph 48372 (\N{HANGUL SYLLABLE BO}) missing from font(s) DejaVu Sans.
  fig.savefig(plot_fname, dpi=250)
/home/codeitDev/miniconda3/envs/codeit/lib/python3.11/site-packages/ultralytics/utils/metrics.py:567: UserWarning: Glyph 47161 (\N{HANGUL SYLLABLE RYEONG}) missing from font(s) DejaVu Sans.
  fig.savefig(plot_fname, dpi=250)
/home/codeitDev/miniconda3/envs/codeit/lib/python3.11/site-packages/ultralytics/utils/metrics.py:567: UserWarning: Glyph 48512 (\N{HANGUL SYLLABLE BU}) missing from font(s) DejaVu Sans.
  fig.savefig(plot_fname, dpi=250)
/home/codeitDev/miniconda3/envs/codeit/lib/python3.11/site-packages/ultralytics/utils/metrics.py:567: UserWarning: Glyph 49828 (\N{HANGUL SYLLABLE SEU}) missing from font(s) DejaVu Sans.
  fig.savefig(plot_fname, dpi=250)
/home/codeitDev/miniconda3/envs/codeit/lib/python3.11/site-packages/ultralytics/utils/metrics.py:567: UserWarni

                   all       1652       5279      0.989      0.927       0.95      0.941
            보령부스파정 5mg         42         42      0.994          1      0.995      0.995
           뮤테란캡슐 100mg        204        204       0.99      0.948      0.994       0.99
           일양하이트린정 2mg         81         81          1      0.871      0.988      0.988
    기넥신에프정(은행엽엑스)(수출용)         31         31      0.988          1      0.995      0.995
     무코스타정(레바미피드)(비매품)         40         40      0.983          1      0.995      0.995
                  알드린정         38         38          1      0.999      0.995      0.995
          뉴로메드정(옥시라세탐)        181        181      0.998          1      0.995      0.995
            타이레놀정500mg         48         48      0.991          1      0.995      0.995
          에어탈정(아세클로페낙)         36         36      0.989          1      0.995      0.995
         삼남건조수산화알루미늄겔정         49         49      0.993          1      0.995      0.995
타이레놀이알서방정(아세트아미노펜)(수출

In [9]:
name_to_cat_id={v:k for k,v in cat_id_to_name.items()}

In [18]:
BASE = './ai06-level1-project'
TEST_IMG_DIR = os.path.join(BASE, "test_cleaned")

In [21]:
PRED_LABEL_DIR = "runs/detect/predict3/labels"
IMAGE_DIR = "runs/detect/predict3"
OUTPUT_CSV = os.path.join(BASE, "original.csv")

cls_to_cat_id = list(cat_id_to_name.keys())

rows = []
annotation_id = 1

txt_files = glob(os.path.join(PRED_LABEL_DIR, "*.txt"))

for txt_path in txt_files:
    fname = os.path.basename(txt_path).replace(".txt", ".jpg")
    img_path = os.path.join(IMAGE_DIR, fname)

    if not os.path.exists(img_path):
        continue

    img = Image.open(img_path)
    W, H = img.size

    image_id = int(os.path.splitext(fname)[0])

    with open(txt_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        cls, x, y, w, h, conf = map(float, line.split())
        cls = int(cls)

        bbox_x = (x - w / 2) * W
        bbox_y = (y - h / 2) * H
        bbox_w = w * W
        bbox_h = h * H

        bbox_x -= 9
        bbox_y -= 9
        bbox_w += 18
        bbox_h += 18

        if bbox_x < 0:
            bbox_w += bbox_x
            bbox_w += bbox_x  # bbox_x가 음수면 그만큼 width 감소
            bbox_x = 0

        if bbox_y < 0:
            bbox_h += bbox_y
            bbox_h += bbox_y
            bbox_y = 0

        if bbox_x + bbox_w > W:
            bbox_w = W - bbox_x

        if bbox_y + bbox_h > H:
            bbox_h = H - bbox_y
        
        p1=(int(bbox_x),int(bbox_y))
        p2=(int(bbox_x+bbox_w),int(bbox_y+bbox_h))
            
        category_id = cls_to_cat_id[cls]

        rows.append({
            "annotation_id": annotation_id,
            "image_id": image_id,
            "category_id": category_id,
            "bbox_x": int(bbox_x),
            "bbox_y": int(bbox_y),
            "bbox_w": int(bbox_w),
            "bbox_h": int(bbox_h),
            "score": round(conf, 4)
        })

        annotation_id += 1

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)
print("CSV saved:", OUTPUT_CSV)

CSV saved: ./ai06-level1-project/original.csv
